# Quick SLM — 05 · SFT training

Fine-tunes the base checkpoint on the packed corpus from notebook 04. Three epochs, LR one
order of magnitude below the pretraining peak, ~2 to 3 hours on an H100.

The trainer is the same object that ran pretraining. The only difference between the stages
is what the dataset yields: pretraining passes `labels = input_ids`, SFT passes labels with
everything outside an assistant turn masked to `-100`.

One thing that is *not* the same, and matters here. `LlamaForCausalLM` returns the mean
cross-entropy over the scored tokens of its micro-batch. The conventional
`loss / grad_accum` therefore averages per-micro-batch means, which equals the true token
mean only when every micro-batch scores the same number of tokens. Pretraining scores every
token, so it does. SFT does not: the fraction of a packed window inside an assistant turn
swings from a few percent to most of it, so the conventional form upweights whichever
windows happen to be mostly context. The trainer scales each micro-batch by its share of the
step's scored tokens instead, which is exactly the token mean and reduces to the
conventional form when the counts are equal.

Val loss is watched every 50 steps. At this corpus size overfitting is the main risk, and
val loss is far more informative than train loss.

> **V1 EDUCATIONAL DEFECT: SFT Schedule Collapse**
> 
> This notebook contains a critical flaw that was documented in the v1 paper regarding the learning rate schedule.
> 
> **The Error**: `grad_accum` was set to 16. Over the truncated 14.7M token SFT corpus, this resulted in only 168 total optimizer steps for the entire 3-epoch run. Because the linear warmup was set to 200 steps, the entire fine-tune was spent in linear warmup, reaching 84% of peak rate on its final step and never decaying at all. 
> 
> **The Lesson**: Always recalculate your total optimizer steps when your corpus size changes (as happened when dedup dropped 43.9% of the planned SFT corpus) to ensure your warmup and cosine decay schedules can complete.



## 1 · Drive and GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

## 2 · Framework

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!pip uninstall -q -y torchvision torchaudio

In [ ]:
EXTRAS = "eval"

# Framework and data live on Drive (no GitHub). Upload the repo folder
# (with pyproject.toml, README.md, and framework/) into DRIVE_ROOT/code once.
# Every notebook installs it from there.
DRIVE_ROOT = "/content/drive/MyDrive/quick-slm"

import subprocess, sys, importlib
from pathlib import Path

root = Path(DRIVE_ROOT)
candidates = [root / "code", root, root / "quick-slm"]
REPO_DIR = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    looked = "\n  ".join(str(p) for p in candidates)
    raise RuntimeError(
        "No pyproject.toml found on Drive. Upload the repo (pyproject.toml, "
        f"README.md, and framework/) to {root / 'code'}, then re-run.\nLooked in:\n  "
        + looked
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# The editable install lands its import hook in site-packages as a .pth file, and
# a .pth is only executed at interpreter startup. This kernel is already running,
# so it never sees the hook and `import v1.quick_slm_trainer` fails until a restart.
# Put framework/ on the path directly and drop the finder caches, so
# the import resolves in this session with no "Restart runtime" step.
framework_dir = REPO_DIR / "framework"
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))
importlib.invalidate_caches()

import v1.quick_slm_trainer as q
print("quick-slm-trainer", q.__version__, "from", Path(q.__file__).parent)
# Refuse to run outside the support window this training version declares in
# training/v1/framework.json. A framework newer than v1's window would not fail
# loudly, it would build a different corpus and the difference would surface as
# unexplained numbers weeks later. If this raises, install the archived build it
# names rather than editing this cell.
if hasattr(q, "require_framework"):
    q.require_framework("v1", REPO_DIR)
else:
    raise RuntimeError(
        "quick-slm-trainer " + q.__version__ + " predates the support-window check; "
        "training v1 requires >=1.0. See SUPPORT.md."
    )


## 3 · Config and corpus

Architecture and tokenizer are inherited from the base checkpoint. Only the optimisation and
run blocks differ, and `sft_v1()` holds them.

`total_steps` comes from the packed corpus, not from a fixed token budget: three epochs over
however many windows notebook 04 produced.

In [ ]:
from v1.quick_slm_trainer import Layout, sft_v1
from v1.quick_slm_trainer.evaluate import corpus_windows
from v1.quick_slm_trainer.tokenizer import load_tokenizer
from v1.quick_slm_trainer.trainer import steps_for_epochs

EPOCHS = 3

layout = Layout().mkdirs_sft()
cfg = sft_v1()
CTX = cfg.data.ctx

BASE_DIR = layout.final_dir  # the pretraining artifact
IDS = layout.sft_bin('train', 'ids')
MASK = layout.sft_bin('train', 'mask')
VAL_IDS = layout.sft_bin('val', 'ids')
VAL_MASK = layout.sft_bin('val', 'mask')

for p in (BASE_DIR / 'config.json', IDS, MASK, VAL_IDS, VAL_MASK):
    if not p.exists():
        raise FileNotFoundError(p)

tok = load_tokenizer(layout.tokenizer_dir, patch=False)

n_windows = corpus_windows(IDS, CTX)
TOTAL_STEPS = steps_for_epochs(n_windows, cfg.run.effective_batch, EPOCHS)
cfg.run.target_tokens = n_windows * CTX * EPOCHS

print(f'base checkpoint : {BASE_DIR}')
print(f'train windows   : {n_windows:,}  ({n_windows * CTX:,} tokens)')
print(f'val windows     : {corpus_windows(VAL_IDS, CTX):,}')
print(f'effective batch : {cfg.run.effective_batch} seq x {CTX} ctx')
print(f'total steps     : {TOTAL_STEPS:,}  ({EPOCHS} epochs)')
print(f'lr              : {cfg.optim.lr_peak:.1e} -> {cfg.optim.lr_min:.1e}, warmup {cfg.optim.warmup_steps}')

## 4 · Load the base model

`load_pretrained` checks the tokenizer's vocabulary against the checkpoint's embedding rows.
SFT is the first stage that emits `<think>` and the ChatML pair, all reserved before
pretraining; a mismatch here means the tokenizer is not the one the base model was trained
with, and every token id in the corpus would be off by the difference.

In [ ]:
import torch

from v1.quick_slm_trainer.model import enable_tf32, load_pretrained, param_count, prepare, set_seed
from v1.quick_slm_trainer.optim import build_optimizer
from v1.quick_slm_trainer.trainer import Trainer, resume_if_possible

DTYPE = torch.bfloat16

set_seed(cfg.run.seed)
enable_tf32()

model = load_pretrained(BASE_DIR, dtype=DTYPE, attn_implementation=cfg.model.attn_implementation, tok=tok)
print(f'parameters : {param_count(model):,}')

model, fp8_active = prepare(model, cfg.run)
optimizer = build_optimizer(model, cfg.optim, device=cfg.run.device)

# Resumes an interrupted SFT run, not the pretraining run: a different checkpoint dir.
state = resume_if_possible(layout.sft_ckpt_dir, model, optimizer)

### Sanity check the mask before spending three hours on it

The single failure mode that produces a plausible loss curve and a useless model is a mask
that scores the wrong tokens. Decode one window and look at what carries loss.

In [ ]:
import numpy as np

from v1.quick_slm_trainer.template import IGNORE_INDEX

ids = np.memmap(IDS, dtype=np.uint16, mode='r')[:CTX].astype(np.int64)
mask = np.memmap(MASK, dtype=np.uint8, mode='r')[:CTX]

scored = ids[mask.astype(bool)]
print(f'scored {mask.sum():,} / {CTX:,} tokens  ({100 * mask.mean():.1f}%)\n')
print('--- SCORED (what the model is graded on) ---')
print(tok.decode(scored[:400]))
print('\n--- CONTEXT (conditioned on, never graded) ---')
print(tok.decode(ids[~mask.astype(bool)][:300]))

## 5 · Loaders and evaluation

In [ ]:
from v1.quick_slm_trainer.loader import make_loader as build_loader
from v1.quick_slm_trainer.sft import MaskedWindowDataset
from v1.quick_slm_trainer.evaluate import evaluate_sft


def make_loader(start_window: int, seed: int):
    ds = MaskedWindowDataset(IDS, MASK, CTX, start_window=start_window, seed=seed)
    return build_loader(
        ds,
        batch_size=cfg.run.micro_batch,
        num_workers=cfg.run.num_workers,
        prefetch_factor=cfg.run.prefetch_factor,
    )


def evaluate():
    # The real held-out split, unlike pretraining's in-corpus probe.
    return evaluate_sft(
        model, VAL_IDS, VAL_MASK, CTX,
        micro_batch=cfg.run.micro_batch, device=cfg.run.device, dtype=DTYPE,
    )


print('val loss before SFT:', evaluate())

## 6 · Train

In [ ]:
trainer = Trainer(
    config=cfg,
    model=model,
    optimizer=optimizer,
    make_loader=make_loader,
    total_steps=TOTAL_STEPS,
    ctx=CTX,
    ckpt_dir=layout.sft_ckpt_dir,
    log_path=layout.logs_dir / 'sft_train.jsonl',
    tokenizer=tok,
    evaluate=evaluate,
    state=state,
    title='Quick SLM — SFT (103M agent)',
    fp8_active=fp8_active,
)

state = trainer.train()
print('\nfinal val loss:', state.val_loss)

## 7 · Probes

Built with the same renderer that packed the corpus. A probe that reformats the prompt measures the mismatch, not the model.

In [ ]:
from v1.quick_slm_trainer.probes import SFT_PROBES, run_probes

model.eval()
for r in run_probes(model, tok, SFT_PROBES, sft=True):
    print('=' * 78)
    print(f'[{r["category"]}]  {r["user_query"]}')
    print(f'expected: {r["expected"]}')
    print('-' * 78)
    print(r['response'])
    print()

## 8 · Final artifact

`training/SFT_README.md` asks for two HuggingFace model IDs: `quick-slm-103m-base` from
notebook 02 and `quick-slm-103m-agent` from here. Publishing both lets a reviewer measure
the SFT contribution cleanly and use the base model in their own ablations.

In [ ]:
from v1.quick_slm_trainer.checkpoint import save_final

final = save_final(layout.sft_final_dir, model, tokenizer=tok, config=cfg)
print('SFT model written to', final)
for f in sorted(final.iterdir()):
    print(f'  {f.name:<28s} {f.stat().st_size / 1e6:>9.2f} MB')